# RL-Book Market Making Experiments (Colab)

Runs all experiments and subparts:
- **Experiment 1**: Inventory-only (1D) — DP vs RL (regular, discrete, distillation)
- **Experiment 2**: (Inventory, Price) 2D — DP vs RL (regular, discrete, distillation)
- **Experiment 3**: (Inventory, Price, Vol) 3D — DP vs RL (regular, discrete, distillation)
- **Experiment 2 Real-World**: Large state space (27,951 states), when RL beats DP

**To run in Colab**: Upload this notebook to [colab.research.google.com](https://colab.research.google.com), or clone the repo in Colab and open `project/experiments_colab.ipynb`.

**A100 GPU**: Runtime → Change runtime type → GPU → A100 (Colab Pro). The setup cell enables TF32 and cuDNN benchmark.

## 1. Setup: Git pull & install dependencies

In [ ]:
# Clone or pull the RL-book repo
import os

REPO_URL = "https://github.com/Sondringsen/RL-book.git"  # or your fork
BRANCH = "sid-discrete-exp-2"  # or "main"

# Colab: clone to /content; local: use workspace
if os.path.exists("/content"):
    REPO_DIR = "/content/RL-book"
    if not os.path.exists(REPO_DIR):
        !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !git pull origin $BRANCH
else:
    # Local: repo root is cwd (if project/ exists) or parent (if we're in project/)
    cwd = os.getcwd()
    if os.path.exists(os.path.join(cwd, "project", "experiment1.py")):
        REPO_DIR = cwd
    elif os.path.exists(os.path.join(cwd, "experiment1.py")):
        REPO_DIR = os.path.abspath(os.path.join(cwd, ".."))
    else:
        REPO_DIR = cwd
    %cd $REPO_DIR
    !git pull origin

PROJECT_DIR = os.path.join(REPO_DIR, "project")
!git status

In [ ]:
# Install dependencies (Colab has numpy, matplotlib; add torch if needed)
!pip install -q torch scipy
print("Dependencies installed.")

In [ ]:
# A100 GPU optimizations (run before experiments)
# In Colab: Runtime → Change runtime type → GPU → A100 (if available)
import torch

if torch.cuda.is_available():
    # cuDNN: benchmark selects fastest algo for fixed input sizes
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False  # faster, slight non-determinism ok
    # TF32 on A100: ~8x faster matmuls with minimal precision loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    # PyTorch 2.0+: high precision for matmuls
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    print(f"GPU: {torch.cuda.get_device_name(0)} — TF32/cuDNN benchmark enabled")
else:
    print("No GPU — running on CPU")

## 2. Experiment 1 — Inventory-only (1D)

In [ ]:
%cd $PROJECT_DIR
import experiment1
experiment1.main()

## 3. Experiment 2 — (Inventory, Price) 2D

In [ ]:
%cd $PROJECT_DIR
import experiment2
experiment2.main()

## 4. Experiment 3 — (Inventory, Price, Vol) 3D

In [ ]:
%cd $PROJECT_DIR
import experiment3
experiment3.main()

## 5. Experiment 2 Real-World — When RL beats DP

In [ ]:
%cd $PROJECT_DIR
import experiment2_realworld
experiment2_realworld.main()

## 6. Git push (optional)

In [ ]:
# Push results. For Colab: use token: git push https://<TOKEN>@github.com/user/repo.git
%cd $REPO_DIR
!git config user.email "colab@example.com" 2>/dev/null || true
!git config user.name "Colab" 2>/dev/null || true
!git add project/results/*.png 2>/dev/null || true
!git status
# To push: !git commit -m 'Colab: run all experiments' && !git push origin $BRANCH